# Lab 1: S3 Vectors Setup for Player Analytics

## Overview

In this foundational lab, you'll set up AWS S3 Vectors infrastructure and create vector embeddings for player analytics data. This creates the vector storage foundation needed for RAG-enhanced AI agents in subsequent labs.

### What You'll Build

**S3 Vectors Infrastructure** with:

- **Vector Bucket**: Dedicated S3 Vectors bucket for gaming data
- **5 Vector Indices**: Separate indices for each dataset type
- **2,902 Vector Embeddings**: Semantic representations of all player data
- **Metadata Storage**: Structured metadata for filtering and retrieval
- **Query Testing**: Verification that semantic search works across all indices

## Prerequisites

To execute this lab you will need:
* AWS credentials configured
* Access to Amazon Bedrock (Cohere Embed English v3)
* Access to AWS S3 Vectors (Preview)

> **How to run:** Click each code cell and press Shift+Enter or the Run button. Execute cells in order from top to bottom.

## Step 1: Environment Setup and Configuration

This section sets up all the dependencies and connections needed for our vector storage infrastructure. We're connecting to S3 Vectors for vector storage, Amazon Bedrock for generating embeddings with Cohere model, and S3 for any additional file operations.

The configuration defines our vector bucket name, data sources, and AWS region.

In [ ]:
import boto3
import json
import numpy as np

VECTOR_BUCKET = "game-embeddings"
AWS_REGION = "us-west-2"

# Get the games data bucket name from SSM Parameter Store
def get_games_data_bucket():
    ssm = boto3.client('ssm')
    try:
        response = ssm.get_parameter(Name='/workshop/games-data-bucket-name')
        bucket_name = response['Parameter']['Value']
        print(f"✅ Games data bucket: {bucket_name}")
        return bucket_name
    except Exception as e:
        print(f"❌ Error getting bucket name from SSM: {e}")
        return None

DATASETS = {
    "player-profiles": "player_profiles.json",
    "lifetime-spend": "lifetime_spend.json",
    "session-history": "session_history.json",
    "player-attributes": "player_attributes.json",
    "items": "items.json"
}

s3 = boto3.client('s3', region_name=AWS_REGION)
s3vectors = boto3.client('s3vectors', region_name=AWS_REGION)
bedrock = boto3.client('bedrock-runtime', region_name=AWS_REGION)

print("✅ Setup complete")

## Step 2: S3 Vectors Infrastructure Creation

This step creates the actual vector storage infrastructure in [AWS S3 Vectors](https://aws.amazon.com/s3/features/vectors/) that will enable Retrieval-Augmented Generation (RAG) in subsequent labs. [RAG is the process of optimizing large language model output](https://aws.amazon.com/what-is/retrieval-augmented-generation/) by referencing an authoritative knowledge base outside of its training data sources before generating a response. This approach makes AI responses more relevant, accurate, and useful by grounding them in specific, up-to-date information. S3 Vectors is a cloud object store with native support to store and query vectors, reducing costs by up to 90% while providing sub-second query performance. We:

- **Create the vector bucket** (`game-embeddings`) - a specialized S3 bucket for vector storage
- **Create 5 separate vector indices** - one for each dataset (player-profiles, lifetime-spend, session-history, player-attributes, items)
- **Configure each index** with 1024 dimensions (Cohere model), float32 data type, and cosine similarity

In [ ]:
# Create S3 Vectors bucket and indices
try:
    # Create vector bucket
    s3vectors.create_vector_bucket(
        vectorBucketName=VECTOR_BUCKET
    )
    print(f"✅ Created vector bucket: {VECTOR_BUCKET}")
except Exception as e:
    if 'ConflictException' in str(e):
        print(f"✅ Vector bucket already exists: {VECTOR_BUCKET}")
    else:
        print(f"❌ Error creating bucket: {e}")

# Create vector indices for each dataset
for dataset_name in DATASETS.keys():
    try:
        s3vectors.create_index(
            vectorBucketName=VECTOR_BUCKET,
            indexName=dataset_name,
            dimension=1024,  # Cohere embed-english-v3 dimension
            dataType='float32',
            distanceMetric='cosine'
        )
        print(f"✅ Created index: {dataset_name}")
    except Exception as e:
        if 'ConflictException' in str(e):
            print(f"✅ Index already exists: {dataset_name}")
        else:
            print(f"❌ Error creating index {dataset_name}: {e}")

print("✅ S3 Vectors infrastructure ready")

## Step 3: Embedding Generation Functions

These functions convert text into numerical representations (vectors) using Amazon Bedrock's Cohere model. [Vector databases](https://aws.amazon.com/what-is/vector-databases/) store these vectors and enable similarity search by comparing numerical patterns between different pieces of content. Embeddings capture semantic meaning - words like "king" and "monarch" will have similar vector values despite different spellings, enabling RAG systems to retrieve contextually relevant information for AI responses. The system supports both single chunks and overlapping chunks for longer texts - chunking with overlap ensures important context isn't lost at chunk boundaries, improving search accuracy.

In [ ]:
def generate_embedding(text):
    max_length = 2048
    overlap = 200
    
    if len(text) <= max_length:
        # Single chunk
        response = bedrock.invoke_model(
            modelId="cohere.embed-english-v3",
            body=json.dumps({
                "texts": [text],
                "input_type": "search_document"
            })
        )
        embedding = json.loads(response["body"].read())["embeddings"][0]
        return np.array(embedding, dtype=np.float32).tolist()
    
    # Multiple overlapping chunks
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_length, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += max_length - overlap
    
    # Get embeddings for all chunks
    response = bedrock.invoke_model(
        modelId="cohere.embed-english-v3",
        body=json.dumps({
            "texts": chunks,
            "input_type": "search_document"
        })
    )
    embeddings = json.loads(response["body"].read())["embeddings"]
    
    # Average the embeddings
    avg_embedding = np.mean(embeddings, axis=0)
    return avg_embedding.astype(np.float32).tolist()

def generate_query_embedding(text):
    """Generate embedding optimized for search queries"""
    response = bedrock.invoke_model(
        modelId="cohere.embed-english-v3",
        body=json.dumps({
            "texts": [text],
            "input_type": "search_query"
        })
    )
    embedding = json.loads(response["body"].read())["embeddings"][0]
    return np.array(embedding, dtype=np.float32).tolist()

def create_text(record, dataset_name):
    parts = [f"Dataset: {dataset_name}"]
    for k, v in record.items():
        parts.append(f"{k}: {v}")
    return "\n".join(parts)

def process_dataset(data, name):
    def clean_record(record):
        cleaned = {}
        for k, v in record.items():
            if v is None or (isinstance(v, list) and len(v) == 0):
                continue
            # Flatten additional_attributes for player-attributes dataset
            if k == "additional_attributes" and isinstance(v, dict):
                # Extract key info from nested structures
                if 'crew_composition' in v:
                    crew_roles = [crew.get('role', '') for crew in v['crew_composition'] if isinstance(crew, dict)]
                    cleaned['crew_roles'] = ','.join(crew_roles)
                if 'skill_upgrades' in v:
                    cleaned['skill_upgrades'] = ','.join(v['skill_upgrades']) if isinstance(v['skill_upgrades'], list) else str(v['skill_upgrades'])
                if 'recent_missions' in v:
                    mission_types = [mission.get('mission_type', '') for mission in v['recent_missions'] if isinstance(mission, dict)]
                    cleaned['mission_types'] = ','.join(mission_types)
                continue
            cleaned[k] = v
        return cleaned
    
    vectors = []
    for i, record in enumerate(data):
        clean_record_data = clean_record(record)
        text = create_text(clean_record_data, name)
        embedding = generate_embedding(text)
        
        # Determine key field
        if name == "items":
            key = clean_record_data["item_id"]
        elif name == "session-history":
            key = clean_record_data["session_id"]
        else:
            key = clean_record_data["player_id"]
        
        if (i + 1) % 500 == 0 or (i + 1) == len(data):
            print(f"    Progress: {i+1}/{len(data)} ({((i+1)/len(data)*100):.1f}%) - {name}")
        
        vectors.append({
            "key": key,
            "embedding": embedding,
            "metadata": clean_record_data
        })
    return vectors

print("✅ Functions defined")

## Step 4: Data Loading and Processing

Load the gaming datasets from S3 and process them into vector embeddings. Each record is converted to structured text, embedded using Cohere's model, and prepared for vector storage with comprehensive metadata preservation.

In [ ]:
# Load datasets from S3
games_bucket = get_games_data_bucket()
datasets = {}
for name, filename in DATASETS.items():
    try:
        response = s3.get_object(Bucket=games_bucket, Key=filename)
        datasets[name] = json.loads(response['Body'].read().decode('utf-8'))
        print(f"Loaded {name}: {len(datasets[name])} records")
    except Exception as e:
        print(f"❌ Error loading {name}: {e}")

print("✅ All datasets loaded from S3")

### Process Datasets into Vector Embeddings

Now we'll convert each dataset into vector embeddings, tracking progress and generating comprehensive metadata for each record.

In [ ]:
# Process all datasets
all_dataset_vectors = {}
completed_datasets = 0

for name, data in datasets.items():
    vectors = process_dataset(data, name)
    all_dataset_vectors[name] = vectors
    
    completed_datasets += 1
    sample_key = vectors[0]['key'] if vectors else 'N/A'
    embedding_dim = len(vectors[0]['embedding']) if vectors else 0
    metadata_fields = len(vectors[0]['metadata']) if vectors else 0
    field_names = list(vectors[0]['metadata'].keys())[:7] if vectors else []
    
    print(f"✅ COMPLETED {name}: {len(vectors)} vectors ({completed_datasets}/{len(datasets)} datasets done)")
    print(f"  Sample key: {sample_key}")
    print(f"  Embedding dim: {embedding_dim}")
    print(f"  Metadata fields: {metadata_fields} - {field_names}")
    print()

print(f"🎉 ALL DATASETS PROCESSED: {sum(len(v) for v in all_dataset_vectors.values())} total vectors")

## Step 5: Local Storage

Save the generated embeddings locally as a backup as a fallback copy. Note that embedding generation can be time-consuming, which is why we use a small sample dataset in this workshop - in production, you may have much larger player datasets requiring longer processing times.

In [ ]:
# Save embeddings to local files
import os
os.makedirs('indices', exist_ok=True)

for name, vectors in all_dataset_vectors.items():
    dataset_dir = f'indices/{name}'
    os.makedirs(dataset_dir, exist_ok=True)
    
    with open(f'{dataset_dir}/embeddings.json', 'w') as f:
        json.dump(vectors, f, indent=2)
    
    print(f"💾 Saved {name}: {len(vectors)} vectors")

print("✅ All embeddings saved to ./indices/")

## Step 6: Vector Upload to S3 Vectors

Upload the processed vectors to S3 Vectors indices using optimized batching for performance. The function handles metadata cleaning, error recovery, and provides progress tracking for large dataset uploads.

In [ ]:
# Put vectors to S3 Vectors
def put_vectors_to_s3(dataset_name, vectors, batch_size=250):
    put_count = 0
    for i in range(0, len(vectors), batch_size):
        batch = [{
            "key": v["key"],
            "data": {"float32": v["embedding"]},
            "metadata": v["metadata"]
        } for v in vectors[i:i + batch_size]]
        
        try:
            s3vectors.put_vectors(vectorBucketName=VECTOR_BUCKET, indexName=dataset_name, vectors=batch)
            put_count += len(batch)
            print(f"  Put {put_count}/{len(vectors)} vectors to {dataset_name}")
        except Exception as e:
            print(f"❌ Error: {e}")
            break
    return put_count

print("✅ Clean put function ready")

### Upload All Datasets to S3 Vectors

Execute the upload process for all processed datasets, tracking progress and completion status for each index.

In [ ]:
# Put all datasets to S3 Vectors
total_put = 0

for dataset_name, vectors in all_dataset_vectors.items():
    print(f"\n📤 Putting vectors to {dataset_name}...")
    put_count = put_vectors_to_s3(dataset_name, vectors)
    total_put += put_count
    print(f"✅ Completed {dataset_name}: {put_count}/{len(vectors)} vectors put")

print(f"\n🎉 PUT VECTORS COMPLETE: {total_put} total vectors put to S3 Vectors")

## Step 7: Testing and Verification

Test the vector storage by running semantic similarity queries against each index to verify embeddings and retrieval functionality. The queries return results with distance scores - lower distances indicate higher similarity between the query and stored vectors. This validates that the complete pipeline from text to searchable vectors works correctly across all gaming datasets.

In [ ]:
# Test S3 Vectors implementation
def test_query_vectors(index_name, query_text, top_k=3):
    print(f"🔍 Testing {index_name}: '{query_text}'")
    try:
        query_embedding = generate_query_embedding(query_text)
        response = s3vectors.query_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=index_name,
            queryVector={"float32": query_embedding},
            topK=top_k,
            returnDistance=True,
            returnMetadata=True
        )
        results = response.get("vectors", [])
        print(f"✅ Found {len(results)} results")
        for i, result in enumerate(results, 1):
            key = result.get("key")
            distance = result.get("distance", 0)
            metadata = result.get("metadata", {})
            print(f"  {i}. {key} (Distance: {distance:.4f})")
            print(f"     Metadata: {metadata}")
    except Exception as e:
        print(f"❌ Error: {e}")
    print()

print("✅ Test function ready")

### Test Player Profiles Index

Query the player-profiles index to verify semantic search works for engagement and demographic data.

In [ ]:
# Test player-profiles index
test_query_vectors("player-profiles", "high engagement PC players in Asia Pacific with low churn")

### Test Lifetime Spend Index

Query the lifetime-spend index to verify semantic search works for spending patterns and transaction data.

In [ ]:
# Test lifetime-spend index
test_query_vectors("lifetime-spend", "high spending many transactions")

### Test Session History Index

Query the session-history index to verify semantic search works for behavioral and activity patterns.

In [ ]:
# Test session-history index
test_query_vectors("session-history", "one hour session 60-70 minutes moderate length")

### Test Player Attributes Index

Query the player-attributes index to verify semantic search works for gameplay context and progression data.

In [ ]:
# Test player-attributes index
test_query_vectors("player-attributes", "hacker with infiltration and sabotage missions with stealth upgrade")

### Test Items Index

Query the items index to verify semantic search works for game items catalog and metadata.

In [ ]:
# Test items index
test_query_vectors("items", "epic weapon with high price for advanced players")

## Congratulations! 🎉

You have successfully:

✅ **Created S3 Vectors infrastructure** with bucket and indices for 5 gaming datasets  
✅ **Generated 2,902 vector embeddings** using Cohere Embed English v3 model  
✅ **Uploaded vectors to S3 Vectors** with proper metadata and batching  
✅ **Tested semantic search** across all indices to verify functionality  
✅ **Saved embeddings locally** for backup and analysis

### Key Concepts Learned:
- **Vector Embeddings**: Converting text data into semantic representations
- **S3 Vectors**: AWS service for storing and querying vector data
- **Semantic Search**: Finding similar content based on meaning, not keywords
- **Metadata Storage**: Preserving structured data alongside vectors

This vector storage foundation enables RAG-enhanced AI agents in subsequent labs to perform semantic queries across your gaming data.